# Phase 2 — Met-Seg Two-Stage CNN Baseline
## Fine-tuning BraTS-METS Pretrained Detector + Segmenter on Cyprus PROTEAS

**Paper:** Sadegheih & Merhof (MICCAI 2024 PRIME)

**Key Advantages over SegResNet:**
- Pretrained on **402 brain metastasis cases** (not gliomas!)
- Two-stage: DenseNet121 detector → DynUNet segmenter
- Designed for **small, scattered metastases**

**Repo:** [xmindflow/Met-Seg](https://github.com/xmindflow/Met-Seg)

In [1]:
# ╔════════════════════════════════════════════════════════════╗
# ║  Met-Seg v3 — FOLD-BY-FOLD (auto-detects completed)      ║
# ╚════════════════════════════════════════════════════════════╝

MODE = 'train_all'   # 'quick_test' | 'train_all'
FOLD = 0             # ignored in train_all (auto-detected)
CV_TYPE = '3fold'

# ╔════════════════════════════════════════════════════════════╗
# ║  HYPERPARAMETERS — Met-Seg v3 (breakthrough)             ║
# ╚════════════════════════════════════════════════════════════╝

CONFIG = {
    'seg_params': {
        'spatial_dims': 3,
        'in_channels': 4,
        'out_channels': 3,
        'kernel_size': [[3,3,3],[3,3,3],[3,3,3],[3,3,3],[3,3,3]],
        'strides': [[1,1,1],[2,2,2],[2,2,2],[2,2,2],[2,2,2]],
        'upsample_kernel_size': [[2,2,2],[2,2,2],[2,2,2],[2,2,2]],
        'deep_supervision': True,
        'deep_supr_num': 3,
        'filters': [32, 64, 128, 256, 320],
        'res_block': True,
        'trans_bias': True,
    },
    'det_params': {
        'spatial_dims': 3,
        'in_channels': 4,
        'out_channels': 1,
        'dropout_prob': 0.2,
    },
    'patch_size': [64, 64, 64],
    'num_samples': 3,               # v3: back to 3 (v2's 5 was too slow)
    'pos_neg_ratio': [2, 1],
    'lr': 2e-4,                     # v3: higher LR for AdamW
    'weight_decay': 1e-5,           # v3: AdamW decoupled weight decay
    'cache_rate': 0.5,
    'batch_size': 2,                # v3: back to 2 (gradient noise helps)
    'num_workers': 0,
    # v3 LR schedule: warmup→flat→step
    'warmup_epochs': 3,             # linear warmup 1e-5 → 2e-4
    'step_epoch_1': 30,             # drop to 5e-5
    'step_epoch_2': 45,             # drop to 1e-5
    'unfreeze_det_epoch': 30,       # unfreeze detector at epoch 30
    'det_lr': 1e-5,                 # detector LR (10x lower)
}

if MODE == 'quick_test':
    CONFIG['epochs'] = 10
    CONFIG['patience'] = 10
    CONFIG['val_interval'] = 2
else:
    CONFIG['epochs'] = 60           # v3: 60 epochs
    CONFIG['patience'] = 25         # v3: generous patience
    CONFIG['val_interval'] = 5      # validate every 5 epochs

REGION_NAMES = ['WT', 'TC', 'ET']

print(f'Mode: {MODE} | CV: {CV_TYPE}')
print(f'Epochs: {CONFIG["epochs"]} | LR: {CONFIG["lr"]} (AdamW, warmup→flat→step)')
print(f'Batch: {CONFIG["batch_size"]} | Samples/vol: {CONFIG["num_samples"]} | Val every {CONFIG["val_interval"]} ep')
print(f'Detector unfreeze at epoch {CONFIG["unfreeze_det_epoch"]}, det_lr={CONFIG["det_lr"]}')
print(f'Strategy: ONE fold per session → stop → relaunch auto-continues')


Mode: train_all | CV: 3fold
Epochs: 60 | LR: 0.0002 (AdamW, warmup→flat→step)
Batch: 2 | Samples/vol: 3 | Val every 5 ep
Detector unfreeze at epoch 30, det_lr=1e-05
Strategy: ONE fold per session → stop → relaunch auto-continues


In [2]:
# Install dependencies
import subprocess, sys
for pkg in ['monai', 'nibabel', 'pytorch-lightning', 'easydict']:
    try:
        __import__(pkg.replace('-', '_'))
    except ImportError:
        subprocess.check_call([sys.executable, '-m', 'pip', 'install', '-q', pkg])
print('Dependencies installed ✅')

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.7/2.7 MB 30.8 MB/s eta 0:00:00
Dependencies installed ✅


In [3]:
import os, shutil
from pathlib import Path

REPO_DIR = Path('/kaggle/working/Met-Seg')
if not REPO_DIR.exists():
    os.system('git clone https://github.com/xmindflow/Met-Seg.git /kaggle/working/Met-Seg')
    print('Cloned Met-Seg repo ✅')
else:
    print('Met-Seg repo already exists ✅')

import sys
sys.path.insert(0, str(REPO_DIR / 'src'))
print(f'Repo at: {REPO_DIR}')

Cloning into '/kaggle/working/Met-Seg'...


Cloned Met-Seg repo ✅
Repo at: /kaggle/working/Met-Seg


In [4]:
import os, subprocess
from pathlib import Path

OUTPUT_ROOT = Path('/kaggle/working/phase2_metseg_outputs')
WEIGHTS_DIR = OUTPUT_ROOT / 'pretrained_weights'
WEIGHTS_DIR.mkdir(parents=True, exist_ok=True)

# Weight files we need
WEIGHT_FILES = {
    'segmentor': 'segmentor_full_modality.ckpt',
    'detector': 'detector_full_modality.ckpt',
}

WEIGHT_URLS = {
    'segmentor': 'https://myfiles.uni-regensburg.de/filr/public-link/file-download/0447879c90b809a80190bbf452df0f6b/120822/-2821463328806488135/segmentor_weight%20%28full%20modality%29.ckpt',
    'detector': 'https://myfiles.uni-regensburg.de/filr/public-link/file-download/0447879c90b809a80190bbf45b1d0f6f/120821/-4880761362689834300/detector_weight%20%28full%20modality%29.ckpt',
}

# ── Strategy 1: Check if weights exist in Kaggle dataset input ──
# (If you uploaded them as a separate dataset called 'metseg-pretrained-weights')
KAGGLE_WEIGHT_DIRS = [
    Path('/kaggle/input/datasets/boufafamoamed/metseg-pretrained-weights'),
    Path('/kaggle/input/metseg-weights'),
    Path('/kaggle/input/met-seg-weights'),
]

def find_weight_in_kaggle(name):
    """Search for weight file in Kaggle dataset inputs."""
    for d in KAGGLE_WEIGHT_DIRS:
        if not d.exists():
            continue
        # Search recursively for .ckpt files
        for f in d.rglob('*.ckpt'):
            if name in f.name.lower() or name in f.stem.lower():
                return f
        # Also check direct children
        for f in d.iterdir():
            if f.suffix == '.ckpt':
                if name in f.name.lower():
                    return f
    return None

for name, filename in WEIGHT_FILES.items():
    dst = WEIGHTS_DIR / filename
    
    if dst.exists() and dst.stat().st_size > 1_000_000:
        print(f'  ✅ {name}: {dst.stat().st_size/1024/1024:.1f} MB (cached)')
        continue
    
    # Strategy 1: Kaggle dataset input
    kaggle_path = find_weight_in_kaggle(name)
    if kaggle_path:
        import shutil
        shutil.copy2(kaggle_path, dst)
        print(f'  ✅ {name}: {dst.stat().st_size/1024/1024:.1f} MB (from Kaggle dataset)')
        continue
    
    # Strategy 2: wget with browser User-Agent (uni-regensburg blocks urllib)
    url = WEIGHT_URLS[name]
    print(f'  Downloading {name} via wget...')
    result = subprocess.run(
        ['wget', '-q', '--no-check-certificate',
         '--user-agent', 'Mozilla/5.0 (X11; Linux x86_64) AppleWebKit/537.36',
         '-O', str(dst), url],
        capture_output=True, text=True, timeout=300
    )
    
    if result.returncode == 0 and dst.exists() and dst.stat().st_size > 1_000_000:
        print(f'  ✅ {name}: {dst.stat().st_size/1024/1024:.1f} MB (downloaded)')
        continue
    
    # Strategy 3: curl as fallback
    print(f'  wget failed, trying curl...')
    result = subprocess.run(
        ['curl', '-L', '-s', '-o', str(dst),
         '-H', 'User-Agent: Mozilla/5.0 (X11; Linux x86_64)',
         url],
        capture_output=True, text=True, timeout=300
    )
    
    if result.returncode == 0 and dst.exists() and dst.stat().st_size > 1_000_000:
        print(f'  ✅ {name}: {dst.stat().st_size/1024/1024:.1f} MB (downloaded via curl)')
        continue
    
    # Failed
    print(f'  ❌ Could not get {name} weights!')
    print(f'     Upload them as a Kaggle dataset called "metseg-pretrained-weights"')
    print(f'     containing: segmentor_full_modality.ckpt and detector_full_modality.ckpt')

print(f'\nWeights dir: {WEIGHTS_DIR}')
for f in WEIGHTS_DIR.glob('*.ckpt'):
    print(f'  {f.name}: {f.stat().st_size/1024/1024:.1f} MB')

  ✅ segmentor: 171.0 MB (from Kaggle dataset)
  ✅ detector: 130.4 MB (from Kaggle dataset)

Weights dir: /kaggle/working/phase2_metseg_outputs/pretrained_weights
  detector_full_modality.ckpt: 130.4 MB
  segmentor_full_modality.ckpt: 171.0 MB


In [5]:
import warnings
# Suppress noisy MONAI and PyTorch warnings for clean logs
warnings.filterwarnings('ignore', message='.*Num foregrounds 0.*')
warnings.filterwarnings('ignore', message='.*non-tuple sequence for multidimensional indexing.*')
warnings.filterwarnings('ignore', message='.*axcodes.*length is smaller.*')

import torch
import torch.nn as nn
import torch.nn.functional as F
import numpy as np
import json
import time
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt
from pathlib import Path
from collections import OrderedDict
from tqdm import tqdm

from monai.networks.nets import DynUNet, DenseNet121
from monai.losses import DiceLoss
from torch.nn import BCEWithLogitsLoss
from monai.data import DataLoader, CacheDataset
from monai.inferers import sliding_window_inference
from monai.metrics import DiceMetric
import monai.transforms as T
from monai.transforms.compose import MapTransform
from monai.utils import ensure_tuple_rep

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f'Device: {device}')
if torch.cuda.is_available():
    print(f'GPU: {torch.cuda.get_device_name(0)}')
    print(f'VRAM: {torch.cuda.get_device_properties(0).total_memory / 1024**3:.1f} GB')

<frozen importlib._bootstrap_external>:1301: FutureWarning: The cuda.cudart module is deprecated and will be removed in a future release, please switch to use the cuda.bindings.runtime module instead.
2026-03-30 09:49:45.192883: E external/local_xla/xla/stream_executor/cuda/cuda_fft.cc:467] Unable to register cuFFT factory: Attempting to register factory for plugin cuFFT when one has already been registered
E0000 00:00:1774864185.405958      23 cuda_dnn.cc:8579] Unable to register cuDNN factory: Attempting to register factory for plugin cuDNN when one has already been registered
E0000 00:00:1774864185.468130      23 cuda_blas.cc:1407] Unable to register cuBLAS factory: Attempting to register factory for plugin cuBLAS when one has already been registered
W0000 00:00:1774864185.972378      23 computation_placer.cc:177] computation placer already registered. Please check linkage and avoid linking the same target more than once.
W0000 00:00:1774864185.972414      23 computation_placer.cc:1

Device: cuda
GPU: Tesla T4
VRAM: 14.6 GB


In [6]:
# ── Data path resolution ──

DATA_ROOT = Path('/kaggle/input/datasets/boufafamoamed/cyprus-proteas-brain-mets')
SYMLINK_DIR = Path('/kaggle/working/nifti_links')

def resolve_path(root, rel):
    p = root / rel
    if p.exists(): return str(p)
    gz = str(p) + '.gz'
    if Path(gz).exists(): return gz
    nii_gz = str(p).replace('.nii.gz', '.nii_gz')
    if Path(nii_gz).exists():
        link = SYMLINK_DIR / rel
        link.parent.mkdir(parents=True, exist_ok=True)
        if not link.exists(): os.symlink(nii_gz, str(link))
        return str(link)
    parent = p.parent
    if parent.exists():
        target = p.name
        for f in parent.iterdir():
            if f.name.lower() == target.lower(): return str(f)
            nii_gz_f = str(f).replace('.nii_gz', '.nii.gz')
            if Path(nii_gz_f).name.lower() == target.lower():
                link = SYMLINK_DIR / rel
                link.parent.mkdir(parents=True, exist_ok=True)
                if not link.exists(): os.symlink(str(f), str(link))
                return str(link)
    raise FileNotFoundError(f'Not found: {rel}')

# Try multiple possible splits file names
splits_file = None
for name in ['data_splits.json', 'cv_splits_3fold.json', 'cv_splits.json']:
    candidate = DATA_ROOT / name
    if candidate.exists():
        splits_file = candidate
        break
if splits_file is None:
    for f in DATA_ROOT.rglob('*splits*.json'):
        splits_file = f; break

with open(splits_file) as f:
    raw_splits = json.load(f)

# Handle nested structure: {'3fold': {'fold_0': ...}} vs flat {'fold_0': ...}
if '3fold' in raw_splits:
    all_splits = raw_splits['3fold']
elif 'fold_0' in raw_splits:
    all_splits = raw_splits
else:
    raise ValueError(f'Cannot find fold data. Keys: {list(raw_splits.keys())}')

print(f'Loaded splits: {splits_file.name}')
print(f'Folds: {list(all_splits.keys())}')

Loaded splits: data_splits.json
Folds: ['fold_0', 'fold_1', 'fold_2']


In [7]:
def build_scan_dicts(scans, label='subset'):
    dicts, skips = [], 0
    for scan in scans:
        try:
            dicts.append({
                'image': [resolve_path(DATA_ROOT, scan['t1']), resolve_path(DATA_ROOT, scan['t1c']),
                          resolve_path(DATA_ROOT, scan['t2']), resolve_path(DATA_ROOT, scan['fla'])],
                'label': resolve_path(DATA_ROOT, scan['mask']),
                'patient_dir': scan['patient_dir'], 'visit': scan['visit'],
            })
        except FileNotFoundError: skips += 1
    if skips: print(f'  Skipped {skips} in {label}')
    return dicts

def get_fold_dicts(fold):
    fd = all_splits[f'fold_{fold}']
    return build_scan_dicts(fd['train_scans'], f'fold{fold}_train'), build_scan_dicts(fd['test_scans'], f'fold{fold}_val')

def get_all_dicts():
    all_d, seen = [], set()
    for fk in all_splits:
        for scan in all_splits[fk]['train_scans'] + all_splits[fk]['test_scans']:
            key = (scan['patient_dir'], scan['visit'])
            if key not in seen:
                seen.add(key)
                try:
                    all_d.append({'image': [resolve_path(DATA_ROOT, scan['t1']), resolve_path(DATA_ROOT, scan['t1c']),
                                            resolve_path(DATA_ROOT, scan['t2']), resolve_path(DATA_ROOT, scan['fla'])],
                                  'label': resolve_path(DATA_ROOT, scan['mask']),
                                  'patient_dir': scan['patient_dir'], 'visit': scan['visit']})
                except FileNotFoundError: pass
    return all_d

train_dicts, val_dicts = get_fold_dicts(0)
print(f'Fold 0: {len(train_dicts)} train | {len(val_dicts)} val')

Fold 0: 114 train | 56 val


In [8]:
# ── Label conversion: Cyprus {0,1,2,3} → BraTS-METS [WT, TC, ET] ──
# Cyprus uses SAME labels as BraTS-METS! No remapping needed!

class ConvertToMultiChannelBratsMetsd(MapTransform):
    """Convert labels to 3-channel [WT, TC, ET] — Met-Seg convention."""
    def __call__(self, data):
        d = dict(data)
        for key in self.key_iterator(d):
            img = d[key]
            if img.ndim == 4 and img.shape[0] == 1:
                img = img.squeeze(0)
            result = [
                (img == 1) | (img == 3) | (img == 2),  # WT
                (img == 1) | (img == 3),                # TC
                img == 3,                                # ET
            ]
            d[key] = torch.stack(result, dim=0).float() if isinstance(img, torch.Tensor) else np.stack(result, axis=0).astype(np.float32)
        return d

print('Label mapping (Cyprus → BraTS-METS):')
print('  Channel 0 (WT) = labels 1+2+3 (all tumor)')
print('  Channel 1 (TC) = labels 1+3 (tumor core)')
print('  Channel 2 (ET) = label 3 (enhancing only)')
print('  NO remapping needed ✅')

Label mapping (Cyprus → BraTS-METS):
  Channel 0 (WT) = labels 1+2+3 (all tumor)
  Channel 1 (TC) = labels 1+3 (tumor core)
  Channel 2 (ET) = label 3 (enhancing only)
  NO remapping needed ✅


In [9]:
# ── Transforms: v3 LIGHT augmentation (flips + mild intensity only) ──
# Heavy augmentation (RandAffine, GaussianNoise, etc.) was REMOVED because
# it confuses pretrained features and caused v2 to underfit.

patch = CONFIG['patch_size']

train_transforms = T.Compose([
    T.LoadImaged(keys=['image', 'label']),
    T.EnsureChannelFirstd(keys=['image', 'label']),
    T.EnsureTyped(keys=['image', 'label']),
    T.Orientationd(keys=['image', 'label'], axcodes='RAS'),
    T.CropForegroundd(keys=['image', 'label'], source_key='image', allow_smaller=True),
    T.NormalizeIntensityd(keys='image', nonzero=True, channel_wise=True),
    ConvertToMultiChannelBratsMetsd(keys=['label']),
    # v3: LIGHT augmentation only (preserves pretrained features)
    T.RandFlipd(keys=['image', 'label'], spatial_axis=[0], prob=0.5),
    T.RandFlipd(keys=['image', 'label'], spatial_axis=[1], prob=0.5),
    T.RandFlipd(keys=['image', 'label'], spatial_axis=[2], prob=0.5),
    T.RandScaleIntensityd(keys='image', factors=0.1, prob=0.3),
    T.SpatialPadd(keys=['image', 'label'], spatial_size=ensure_tuple_rep(patch, 3)),
    T.RandCropByPosNegLabeld(keys=['image', 'label'], label_key='label',
        spatial_size=ensure_tuple_rep(patch, 3),
        pos=CONFIG['pos_neg_ratio'][0], neg=CONFIG['pos_neg_ratio'][1],
        num_samples=CONFIG['num_samples'], image_key='image', image_threshold=0),
    T.EnsureTyped(keys=['image', 'label'], dtype=torch.float32),
])

val_transforms = T.Compose([
    T.LoadImaged(keys=['image', 'label']),
    T.EnsureChannelFirstd(keys=['image', 'label']),
    T.EnsureTyped(keys=['image', 'label']),
    T.Orientationd(keys=['image', 'label'], axcodes='RAS'),
    T.CropForegroundd(keys=['image', 'label'], source_key='image', allow_smaller=True),
    T.NormalizeIntensityd(keys='image', nonzero=True, channel_wise=True),
    ConvertToMultiChannelBratsMetsd(keys=['label']),
    T.EnsureTyped(keys=['image', 'label'], dtype=torch.float32),
])

print('Transforms defined ✅')
print(f'  v3: Light augmentation (flips + mild intensity)')
print(f'  {CONFIG["num_samples"]} random crops per volume')

Transforms defined ✅
  v3: Light augmentation (flips + mild intensity)
  3 random crops per volume


/usr/local/lib/python3.12/dist-packages/monai/utils/deprecate_utils.py:321: FutureWarning: monai.transforms.spatial.dictionary Orientationd.__init__:labels: Current default value of argument `labels=(('L', 'R'), ('P', 'A'), ('I', 'S'))` was changed in version None from `labels=(('L', 'R'), ('P', 'A'), ('I', 'S'))` to `labels=None`. Default value changed to None meaning that the transform now uses the 'space' of a meta-tensor, if applicable, to determine appropriate axis labels.
  warn_deprecated(argname, msg, warning_category)


In [10]:
# ── Model creation & weight loading ──

def create_segmenter():
    return DynUNet(**CONFIG['seg_params'])

def create_detector():
    return DenseNet121(**CONFIG['det_params'])

def load_lightning_weights(model, ckpt_path, prefix='model.'):
    ckpt = torch.load(ckpt_path, map_location='cpu', weights_only=False)
    sd = ckpt.get('state_dict', ckpt)
    new_sd = OrderedDict()
    for k, v in sd.items():
        new_key = k[len(prefix):] if k.startswith(prefix) else k
        new_sd[new_key] = v
    missing, unexpected = model.load_state_dict(new_sd, strict=False)
    return missing, unexpected

# ── Find weight files ──
# Priority: WEIGHTS_DIR (from cell 4) → Kaggle input datasets
def find_weight(name):
    """Find a weight file by name."""
    # Check WEIGHTS_DIR first (from cell 4 download)
    candidate = WEIGHTS_DIR / f'{name}_full_modality.ckpt'
    if candidate.exists() and candidate.stat().st_size > 1_000_000:
        return candidate
    
    # Search Kaggle input directories
    import glob
    search_patterns = [
        f'/kaggle/input/metseg-pretrained-weights/**/{name}*.ckpt',
        f'/kaggle/input/metseg-pretrained-weights/**/*{name}*.ckpt',
        f'/kaggle/input/met-seg*/**/{name}*.ckpt',
        f'/kaggle/input/*metseg*/**/*{name}*.ckpt',
        f'/kaggle/input/**/{name}*modality*.ckpt',
    ]
    for pattern in search_patterns:
        matches = glob.glob(pattern, recursive=True)
        for m in matches:
            if os.path.getsize(m) > 1_000_000:
                return Path(m)
    
    return None

seg_weights = find_weight('segmentor')
det_weights = find_weight('detector')

print(f'Segmenter weights: {seg_weights}')
print(f'Detector weights:  {det_weights}')

# Test model creation + weight loading
seg_test = create_segmenter()
det_test = create_detector()

if seg_weights:
    print('\nLoading segmenter weights...')
    sm, su = load_lightning_weights(seg_test, seg_weights)
    print(f'  Missing: {len(sm)} | Unexpected: {len(su)}')
    if sm: print(f'  Missing keys: {sm[:3]}...')
else:
    print('\n⚠️ Segmenter weights not found — training from scratch')

if det_weights:
    print('Loading detector weights...')
    dm, du = load_lightning_weights(det_test, det_weights)
    print(f'  Missing: {len(dm)} | Unexpected: {len(du)}')
else:
    print('⚠️ Detector weights not found — training from scratch')

sp = sum(p.numel() for p in seg_test.parameters())
dp = sum(p.numel() for p in det_test.parameters())
print(f'\nSegmenter (DynUNet): {sp:,} params ({sp/1e6:.1f}M)')
print(f'Detector (DenseNet121): {dp:,} params ({dp/1e6:.1f}M)')
print(f'Total: {(sp+dp):,} params ({(sp+dp)/1e6:.1f}M)')

has_pretrained = seg_weights is not None and det_weights is not None
if has_pretrained:
    print('\n✅ BraTS-METS pretrained weights loaded successfully!')
else:
    print('\n⚠️ Some weights missing — check that you added "metseg-pretrained-weights" dataset')

del seg_test, det_test


Segmenter weights: /kaggle/working/phase2_metseg_outputs/pretrained_weights/segmentor_full_modality.ckpt
Detector weights:  /kaggle/working/phase2_metseg_outputs/pretrained_weights/detector_full_modality.ckpt

Loading segmenter weights...
  Missing: 0 | Unexpected: 727
Loading detector weights...
  Missing: 0 | Unexpected: 0

Segmenter (DynUNet): 16,674,764 params (16.7M)
Detector (DenseNet121): 11,309,505 params (11.3M)
Total: 27,984,269 params (28.0M)

✅ BraTS-METS pretrained weights loaded successfully!


In [11]:
# ── Embedding hook for DynUNet bottleneck ──

_embedding_storage = {}

def _hook_fn(module, input, output):
    if isinstance(output, (list, tuple)): feat = output[0]
    else: feat = output
    _embedding_storage['feat'] = feat.detach()

def register_embedding_hook(model):
    # DynUNet: hook the last downsample block (320 channels)
    target = None
    if hasattr(model, 'downsamples'):
        target = model.downsamples[-1]
    if target is not None:
        hook = target.register_forward_hook(_hook_fn)
        print(f'  Hook registered on last downsample block ✅')
        return hook
    else:
        print('  ⚠️ Listing modules to find bottleneck:')
        for name, _ in model.named_children():
            print(f'    {name}')
        return None

print('Embedding hook ready')

Embedding hook ready


In [12]:
def get_lr_for_epoch(epoch, config):
    """v3 LR schedule: warmup → flat → step down."""
    warmup = config['warmup_epochs']
    base_lr = config['lr']
    if epoch < warmup:
        return 1e-5 + (base_lr - 1e-5) * (epoch / warmup)
    elif epoch < config['step_epoch_1']:
        return base_lr
    elif epoch < config['step_epoch_2']:
        return 5e-5
    else:
        return 1e-5

def train_fold(fold):
    """Train one fold with AdamW, step LR, detector unfreeze — v3."""
    
    ckpt_dir = OUTPUT_ROOT / 'checkpoints'; ckpt_dir.mkdir(parents=True, exist_ok=True)
    best_path = ckpt_dir / f'metseg_fold{fold}_best.pth'
    latest_path = ckpt_dir / f'metseg_fold{fold}_latest.pth'
    
    # ── Step 1: Check if fold is COMPLETE ──
    if best_path.exists() and latest_path.exists():
        latest_ckpt = torch.load(latest_path, map_location='cpu', weights_only=False)
        if latest_ckpt.get('epoch', -1) >= CONFIG['epochs'] - 1:
            bd = latest_ckpt.get('best_dice', 0)
            print(f'  ✅ Fold {fold} already complete (Dice={bd:.4f}), skipping')
            seg_model = create_segmenter().to(device)
            best_ckpt = torch.load(best_path, map_location=device, weights_only=False)
            seg_model.load_state_dict(best_ckpt['seg_state_dict'])
            return seg_model, bd, best_ckpt.get('metrics_history', {}), True
    
    # ── Step 2: Build data loaders ──
    train_dicts, val_dicts = get_fold_dicts(fold)
    print(f'  Train: {len(train_dicts)} | Val: {len(val_dicts)}')
    
    train_ds = CacheDataset(train_dicts, train_transforms, cache_rate=CONFIG['cache_rate'], num_workers=CONFIG['num_workers'])
    val_ds = CacheDataset(val_dicts, val_transforms, cache_rate=1.0, num_workers=CONFIG['num_workers'])
    train_loader = DataLoader(train_ds, batch_size=CONFIG['batch_size'], shuffle=True, num_workers=CONFIG['num_workers'], pin_memory=True)
    val_loader = DataLoader(val_ds, batch_size=1, shuffle=False, num_workers=CONFIG['num_workers'])
    
    # ── Step 3: Create models and load pretrained weights ──
    seg_model = create_segmenter().to(device)
    det_model = create_detector().to(device)
    
    if seg_weights:
        load_lightning_weights(seg_model, seg_weights)
        print('  Loaded segmenter pretrained weights ✅')
    if det_weights:
        load_lightning_weights(det_model, det_weights)
        print('  Loaded detector pretrained weights ✅')
    
    det_model.eval()
    for p in det_model.parameters(): p.requires_grad = False
    det_unfrozen = False
    
    # ── Step 4: AdamW optimizer (v3) ──
    optimizer = torch.optim.AdamW(seg_model.parameters(), lr=CONFIG['lr'],
        weight_decay=CONFIG['weight_decay'], betas=(0.9, 0.999))
    
    dice_loss_fn = DiceLoss(sigmoid=True, smooth_nr=0, smooth_dr=1e-5)
    bce_loss_fn = BCEWithLogitsLoss()
    scaler = torch.amp.GradScaler('cuda')
    dice_metric = DiceMetric(include_background=True, reduction='mean_batch')
    
    best_dice, patience_ctr = 0.0, 0
    start_epoch = 0
    metrics_log = {'train_loss': [], 'val_dice': [], 'val_per_region': [], 'lr': []}
    
    # ── Step 5: Check for RESUME from latest checkpoint ──
    if latest_path.exists():
        print(f'  🔄 Found latest checkpoint, attempting resume...')
        ckpt = torch.load(latest_path, map_location=device, weights_only=False)
        seg_model.load_state_dict(ckpt['seg_state_dict'])
        optimizer.load_state_dict(ckpt['optimizer_state_dict'])
        start_epoch = ckpt['epoch'] + 1
        best_dice = ckpt.get('best_dice', 0.0)
        metrics_log = ckpt.get('metrics_history', metrics_log)
        patience_ctr = ckpt.get('patience_ctr', 0)
        det_unfrozen = ckpt.get('det_unfrozen', False)
        if det_unfrozen and 'det_state_dict' in ckpt:
            det_model.load_state_dict(ckpt['det_state_dict'])
            for p in det_model.parameters(): p.requires_grad = True
            optimizer.add_param_group({'params': det_model.parameters(), 'lr': CONFIG['det_lr'], 'weight_decay': CONFIG['weight_decay']})
        print(f'  🔄 Resuming fold {fold} from epoch {start_epoch} (best so far: {best_dice:.4f})')
    
    # ── Step 6: GPU setup ──
    n_gpus = torch.cuda.device_count()
    print(f'  Using GPU 0 ({torch.cuda.get_device_name(0)}) | {n_gpus} available')
    
    def get_state_dict(model):
        return model.module.state_dict() if hasattr(model, 'module') else model.state_dict()
    
    t0 = time.time()
    
    # ── Step 7: Training loop ──
    for epoch in range(start_epoch, CONFIG['epochs']):
        # v3: Manual LR schedule (warmup → flat → step)
        current_lr = get_lr_for_epoch(epoch, CONFIG)
        for pg in optimizer.param_groups:
            if pg.get('lr', 0) > CONFIG.get('det_lr', 1e-5) or not det_unfrozen:
                pg['lr'] = current_lr
        
        # v3: Unfreeze detector at specified epoch
        if epoch >= CONFIG['unfreeze_det_epoch'] and not det_unfrozen:
            for p in det_model.parameters(): p.requires_grad = True
            optimizer.add_param_group({'params': det_model.parameters(), 'lr': CONFIG['det_lr'], 'weight_decay': CONFIG['weight_decay']})
            det_unfrozen = True
            print(f'  🔓 Detector unfrozen at epoch {epoch} (det_lr={CONFIG["det_lr"]:.1e})')
        
        seg_model.train()
        if det_unfrozen: det_model.train()
        epoch_loss, n_steps = 0.0, 0
        for batch in train_loader:
            images = batch['image'].to(device)
            labels = batch['label'].to(device)
            
            # Stage 1: Detector
            if det_unfrozen:
                det_pred = det_model(images)
            else:
                with torch.no_grad():
                    det_pred = det_model(images)
            tumor_mask = torch.sigmoid(det_pred.squeeze(-1)) > 0.3
            if tumor_mask.sum() == 0:
                tumor_mask = torch.ones(len(images), dtype=torch.bool, device=device)
            chosen_img = images[tumor_mask]
            chosen_lbl = labels[tumor_mask]
            
            optimizer.zero_grad()
            with torch.amp.autocast('cuda'):
                outputs = seg_model(chosen_img)
                if isinstance(outputs, (list, tuple)):
                    loss = sum(0.5**i * (dice_loss_fn(p, chosen_lbl) + bce_loss_fn(p, chosen_lbl)) for i, p in enumerate(outputs))
                elif outputs.dim() == 6:
                    preds = torch.unbind(outputs, dim=1)
                    loss = sum(0.5**i * (dice_loss_fn(p, chosen_lbl) + bce_loss_fn(p, chosen_lbl)) for i, p in enumerate(preds))
                else:
                    loss = dice_loss_fn(outputs, chosen_lbl) + bce_loss_fn(outputs, chosen_lbl)
            scaler.scale(loss).backward()
            scaler.unscale_(optimizer)
            torch.nn.utils.clip_grad_norm_(seg_model.parameters(), max_norm=12.0)
            scaler.step(optimizer)
            scaler.update()
            epoch_loss += loss.item()
            n_steps += 1
        
        avg_loss = epoch_loss / max(n_steps, 1)
        metrics_log['train_loss'].append(avg_loss)
        metrics_log['lr'].append(current_lr)
        
        # ── Validation ──
        if (epoch + 1) % CONFIG['val_interval'] == 0 or epoch == CONFIG['epochs'] - 1:
            seg_model.eval()
            dice_metric.reset()
            with torch.no_grad():
                for vb in val_loader:
                    vi = vb['image'].to(device)
                    vl = vb['label'].to(device)
                    vo = sliding_window_inference(vi, CONFIG['patch_size'], 4, seg_model, overlap=0.5, mode='gaussian')
                    if isinstance(vo, (list, tuple)): vo = vo[0]
                    if vo.dim() == 6: vo = vo[:, 0]
                    vp = (torch.sigmoid(vo) > 0.5).float()
                    dice_metric(vp, vl)
            dv = dice_metric.aggregate()
            md = dv.mean().item()
            pr = [dv[i].item() for i in range(len(REGION_NAMES))]
            metrics_log['val_dice'].append(md)
            metrics_log['val_per_region'].append(pr)
            elapsed = (time.time() - t0) / 60
            rs = ' '.join([f'{n}={v:.3f}' for n, v in zip(REGION_NAMES, pr)])
            print(f'Epoch {epoch:3d}/{CONFIG["epochs"]-1} | Loss: {avg_loss:.4f} | Dice: {md:.4f} ({rs}) | LR: {current_lr:.1e} | {elapsed:.1f} min')
            if md > best_dice:
                best_dice = md; patience_ctr = 0
                torch.save({
                    'epoch': epoch, 'best_dice': best_dice,
                    'seg_state_dict': get_state_dict(seg_model),
                    'det_state_dict': get_state_dict(det_model),
                    'optimizer_state_dict': optimizer.state_dict(),
                    'metrics_history': metrics_log,
                    'det_unfrozen': det_unfrozen,
                    'config': CONFIG,
                }, best_path)
                print(f'  ✅ New best! Saved.')
            else:
                patience_ctr += 1
                if patience_ctr >= CONFIG['patience']:
                    print(f'  Early stopping at epoch {epoch}.'); break
        else:
            elapsed = (time.time() - t0) / 60
            print(f'Epoch {epoch:3d}/{CONFIG["epochs"]-1} | Loss: {avg_loss:.4f} | LR: {current_lr:.1e} | {elapsed:.1f} min')
        
        # Save LATEST checkpoint every epoch (overwrite)
        torch.save({
            'epoch': epoch, 'best_dice': best_dice,
            'seg_state_dict': get_state_dict(seg_model),
            'det_state_dict': get_state_dict(det_model),
            'optimizer_state_dict': optimizer.state_dict(),
            'metrics_history': metrics_log,
            'patience_ctr': patience_ctr,
            'det_unfrozen': det_unfrozen,
            'config': CONFIG,
        }, latest_path)
    
    elapsed = (time.time() - t0) / 60
    print(f'\n  Fold {fold} complete: Best Dice = {best_dice:.4f} | Time = {elapsed:.1f} min')
    
    # ── Save figures and metrics ──
    fig_dir = OUTPUT_ROOT / 'figures'; fig_dir.mkdir(parents=True, exist_ok=True)
    fig, axes = plt.subplots(1, 3, figsize=(18, 5))
    axes[0].plot(metrics_log['train_loss']); axes[0].set_title(f'Loss (Fold {fold})'); axes[0].set_xlabel('Epoch')
    if metrics_log['val_dice']:
        axes[1].plot(metrics_log['val_dice'], label='Mean Dice', marker='o'); axes[1].legend()
    axes[1].set_title(f'Val Dice (Fold {fold})'); axes[1].set_xlabel('Validation Step')
    if metrics_log['lr']:
        axes[2].plot(metrics_log['lr']); axes[2].set_title(f'LR Schedule (warmup→flat→step)')
        axes[2].set_xlabel('Epoch'); axes[2].set_ylabel('LR')
    plt.tight_layout(); plt.savefig(fig_dir / f'metseg_fold{fold}_training_curves.png', dpi=150); plt.close()
    
    met_dir = OUTPUT_ROOT / 'metrics'; met_dir.mkdir(parents=True, exist_ok=True)
    with open(met_dir / f'metseg_fold{fold}_metrics.json', 'w') as f: json.dump(metrics_log, f, indent=2)
    
    if hasattr(seg_model, 'module'):
        seg_model = seg_model.module
    
    return seg_model, best_dice, metrics_log, False


In [13]:
def extract_embeddings(model, fold):
    model.eval(); model.to(device)
    global _embedding_storage; _embedding_storage = {}
    emb_hook = register_embedding_hook(model)
    all_dicts = get_all_dicts()
    print(f'  Extracting from {len(all_dicts)} scans...')
    emb_ds = CacheDataset(all_dicts, val_transforms, cache_rate=CONFIG['cache_rate'], num_workers=CONFIG['num_workers'])
    emb_loader = DataLoader(emb_ds, batch_size=1, shuffle=False, num_workers=CONFIG['num_workers'])
    embeddings = {}
    with torch.no_grad():
        for i, batch in enumerate(tqdm(emb_loader, desc='Extracting')):
            images = batch['image'].to(device)
            patient = batch['patient_dir'][0]; visit = batch['visit'][0]
            key = f'{patient}__{visit}'
            _ = sliding_window_inference(images, CONFIG['patch_size'], 4, model, overlap=0.5, mode='gaussian')
            if 'feat' in _embedding_storage:
                feat = _embedding_storage['feat']
                emb = F.adaptive_avg_pool3d(feat, 1).flatten()
                embeddings[key] = emb.cpu().numpy()
    if emb_hook is not None: emb_hook.remove()
    if len(embeddings) == 0: print('  ⚠️ No embeddings!'); return
    emb_dir = OUTPUT_ROOT / 'embeddings'; emb_dir.mkdir(parents=True, exist_ok=True)
    emb_dim = list(embeddings.values())[0].shape[0]
    np.savez(emb_dir / f'cnn_metseg_embeddings_fold{fold}.npz', **embeddings)
    meta = {k: {'patient_dir': k.split('__')[0], 'visit': k.split('__')[1]} for k in embeddings}
    with open(emb_dir / f'cnn_metseg_embeddings_fold{fold}_meta.json', 'w') as f: json.dump(meta, f, indent=2)
    print(f'  ✅ {len(embeddings)} embeddings × {emb_dim}-dim saved')

In [14]:
# ╔════════════════════════════════════════════════════════════╗
# ║  MAIN — v3 FOLD-BY-FOLD (one fold per session)           ║
# ║  Relaunch auto-detects completed folds and continues     ║
# ╚════════════════════════════════════════════════════════════╝

import torch
n_gpus = torch.cuda.device_count()
print(f'GPUs available: {n_gpus}')
for i in range(n_gpus):
    print(f'  GPU {i}: {torch.cuda.get_device_name(i)} ({torch.cuda.get_device_properties(i).total_memory / 1e9:.1f} GB)')

# ── Find the NEXT fold to train ──
ckpt_dir = OUTPUT_ROOT / 'checkpoints'
ckpt_dir.mkdir(parents=True, exist_ok=True)

completed_folds = {}
target_fold = None
for f in [0, 1, 2]:
    best_p = ckpt_dir / f'metseg_fold{f}_best.pth'
    latest_p = ckpt_dir / f'metseg_fold{f}_latest.pth'
    if best_p.exists() and latest_p.exists():
        ckpt = torch.load(latest_p, map_location='cpu', weights_only=False)
        if ckpt.get('epoch', -1) >= CONFIG['epochs'] - 1:
            completed_folds[f] = ckpt.get('best_dice', 0)
            print(f'  ✅ Fold {f}: COMPLETE (Dice={completed_folds[f]:.4f})')
            continue
        else:
            print(f'  🔄 Fold {f}: PARTIAL (epoch {ckpt.get("epoch", 0)}/{CONFIG["epochs"]-1})')
    else:
        print(f'  🆕 Fold {f}: NOT STARTED')
    if target_fold is None:
        target_fold = f

if target_fold is None:
    print('\n' + '=' * 60)
    print('  🎉 ALL 3 FOLDS COMPLETE!')
    print('=' * 60)
    for f, d in completed_folds.items():
        print(f'  Fold {f}: Dice={d:.4f}')
    mean_dice = sum(completed_folds.values()) / len(completed_folds)
    print(f'  Mean Dice: {mean_dice:.4f}')
    
    # Extract embeddings for all folds
    for f in [0, 1, 2]:
        emb_path = OUTPUT_ROOT / f'embeddings/cnn_metseg_embeddings_fold{f}.npz'
        if not emb_path.exists():
            print(f'\n  Extracting embeddings for fold {f}...')
            seg_model = create_segmenter().to(device)
            best_ckpt = torch.load(ckpt_dir / f'metseg_fold{f}_best.pth', map_location=device, weights_only=False)
            seg_model.load_state_dict(best_ckpt['seg_state_dict'])
            extract_embeddings(seg_model, f)
            del seg_model; torch.cuda.empty_cache()
else:
    print(f'\n  ▶▶ Training FOLD {target_fold} this session')
    print('=' * 60)
    print(f'  Training Fold {target_fold} | v3 | {CONFIG["epochs"]} epochs')
    print('=' * 60)
    
    model, best_dice, metrics, was_skipped = train_fold(target_fold)
    
    # Extract embeddings for this fold
    best_path = ckpt_dir / f'metseg_fold{target_fold}_best.pth'
    if best_path.exists():
        ckpt = torch.load(best_path, map_location=device, weights_only=False)
        model.load_state_dict(ckpt['seg_state_dict'])
    print(f'\n  Extracting embeddings for fold {target_fold}...')
    extract_embeddings(model, target_fold)
    
    completed_folds[target_fold] = best_dice
    remaining = [f for f in [0,1,2] if f not in completed_folds]
    
    print('\n' + '=' * 60)
    print(f'  ✅ Fold {target_fold} DONE — Dice={best_dice:.4f}')
    print(f'  Completed: {list(completed_folds.keys())} | Remaining: {remaining}')
    if remaining:
        print(f'  → Relaunch notebook to train fold {remaining[0]}')
    else:
        print(f'  🎉 ALL FOLDS COMPLETE!')
        mean_dice = sum(completed_folds.values()) / len(completed_folds)
        print(f'  Mean Dice: {mean_dice:.4f}')
    print('=' * 60)

print(f'\n  Files saved to {OUTPUT_ROOT}:')
for p in sorted(OUTPUT_ROOT.rglob('*')):
    if p.is_file():
        size_mb = p.stat().st_size / (1024 * 1024)
        print(f'    {p.relative_to(OUTPUT_ROOT)} ({size_mb:.1f} MB)')
total_mb = sum(p.stat().st_size for p in OUTPUT_ROOT.rglob('*') if p.is_file()) / (1024*1024)
print(f'\n  Total output size: {total_mb:.1f} MB / 19,000 MB limit')


GPUs available: 2
  GPU 0: Tesla T4 (15.6 GB)
  GPU 1: Tesla T4 (15.6 GB)
  🆕 Fold 0: NOT STARTED
  🆕 Fold 1: NOT STARTED
  🆕 Fold 2: NOT STARTED

  ▶▶ Training FOLD 0 this session
  Training Fold 0 | v3 | 60 epochs
  Train: 114 | Val: 56


Loading dataset: 100%|██████████| 56/56 [01:50<00:00,  1.98s/it]


  Loaded segmenter pretrained weights ✅
  Loaded detector pretrained weights ✅
  Using GPU 0 (Tesla T4) | 2 available
Epoch   0/59 | Loss: 2.8377 | LR: 1.0e-05 | 2.5 min
Epoch   1/59 | Loss: 2.3460 | LR: 7.3e-05 | 5.0 min
Epoch   2/59 | Loss: 2.1521 | LR: 1.4e-04 | 7.3 min
Epoch   3/59 | Loss: 2.0262 | LR: 2.0e-04 | 9.7 min
Epoch   4/59 | Loss: 1.9644 | Dice: 0.3642 (WT=0.393 TC=0.353 ET=0.346) | LR: 2.0e-04 | 18.5 min
  ✅ New best! Saved.
Epoch   5/59 | Loss: 1.8997 | LR: 2.0e-04 | 20.9 min
Epoch   6/59 | Loss: 1.8424 | LR: 2.0e-04 | 23.2 min
Epoch   7/59 | Loss: 1.7417 | LR: 2.0e-04 | 25.6 min
Epoch   8/59 | Loss: 1.7007 | LR: 2.0e-04 | 27.9 min
Epoch   9/59 | Loss: 1.6613 | Dice: 0.3457 (WT=0.379 TC=0.330 ET=0.328) | LR: 2.0e-04 | 36.6 min
Epoch  10/59 | Loss: 1.6189 | LR: 2.0e-04 | 38.9 min
Epoch  11/59 | Loss: 1.6464 | LR: 2.0e-04 | 41.3 min
Epoch  12/59 | Loss: 1.5975 | LR: 2.0e-04 | 43.6 min
Epoch  13/59 | Loss: 1.5085 | LR: 2.0e-04 | 45.9 min
Epoch  14/59 | Loss: 1.5182 | Dice:

Extracting: 100%|██████████| 170/170 [22:20<00:00,  7.88s/it]


  ✅ 170 embeddings × 1024-dim saved

  ✅ Fold 0 DONE — Dice=0.5316
  Completed: [0] | Remaining: [1, 2]
  → Relaunch notebook to train fold 1

  Files saved to /kaggle/working/phase2_metseg_outputs:
    checkpoints/metseg_fold0_best.pth (234.7 MB)
    checkpoints/metseg_fold0_latest.pth (234.7 MB)
    embeddings/cnn_metseg_embeddings_fold0.npz (0.7 MB)
    embeddings/cnn_metseg_embeddings_fold0_meta.json (0.0 MB)
    figures/metseg_fold0_training_curves.png (0.1 MB)
    metrics/metseg_fold0_metrics.json (0.0 MB)
    pretrained_weights/detector_full_modality.ckpt (130.4 MB)
    pretrained_weights/segmentor_full_modality.ckpt (171.0 MB)

  Total output size: 771.7 MB / 19,000 MB limit
